
# Notebook 02 – Data Preparation & Recruitment Modelling

## Purpose

Prepare 2025/26 Big Five European league performance data and develop an interpretable Defensive Controller recruitment model for Manchester United.

## Inputs

- Original 2025/26 player dataset
- Five SofaScore league performance datasets
- SofaScore player metadata

## Outputs

- `data/processed/defensive_controller_candidates_2025_26.csv`
- `data/processed/recruitment_pool_2025_26.csv`
- `data/processed/defensive_controller_shortlist_2025_26.csv`

## Scope

This notebook covers data validation, positional enrichment, transfer-player aggregation, feature engineering, performance scoring, sensitivity analysis, recruitment filtering and market-value analysis.

Detailed visualisations and final recruitment recommendations will be presented separately.


### 1.1 Import Libraries

Import the libraries required for data preparation.

- `Path` from `pathlib` is used to construct file paths in a platform-independent way.
- `pandas` is used for loading, inspecting and transforming tabular player data.

In [1]:
from pathlib import Path
import pandas as pd

### 1.2 Define Project Paths

Define the project root and construct the path to the original 2025/26 player dataset.

Using paths relative to the project directory rather than absolute paths makes the notebook reproducible if the repository is moved to another computer.

In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "2025-26"
    / "players_data-2025_2026.csv"
)

### 1.3 Load the Initial Dataset

Load the original 2025/26 player dataset into a pandas DataFrame.

The following cells inspect its size and available columns before deciding whether it contains the metrics required to model the Defensive Controller role.

In [3]:
df = pd.read_csv(RAW_DATA_PATH)

### 1.4 Inspect Dataset Structure

Inspect the number of rows and columns and list all available variables.

This provides an initial understanding of the dataset and allows us to determine whether the required passing, possession and defensive statistics are available.

In [4]:
len(df)

2839

In [5]:
df.shape

(2839, 102)

In [6]:
for column in df.columns:
    print(column)

Rk
Player
Nation
Pos
Squad
Comp
Age
Born
MP
Starts
Min
90s
Gls
Ast
G+A
G-PK
PK
PKatt
CrdY
CrdR
G+A-PK
Rk_stats_keeper
Nation_stats_keeper
Pos_stats_keeper
Comp_stats_keeper
Age_stats_keeper
Born_stats_keeper
MP_stats_keeper
Starts_stats_keeper
Min_stats_keeper
90s_stats_keeper
GA
GA90
SoTA
Saves
Save%
W
D
L
CS
CS%
PKatt_stats_keeper
PKA
PKsv
PKm
Rk_stats_shooting
Nation_stats_shooting
Pos_stats_shooting
Comp_stats_shooting
Age_stats_shooting
Born_stats_shooting
90s_stats_shooting
Gls_stats_shooting
Sh
SoT
SoT%
Sh/90
SoT/90
G/Sh
G/SoT
PK_stats_shooting
PKatt_stats_shooting
Rk_stats_playing_time
Nation_stats_playing_time
Pos_stats_playing_time
Comp_stats_playing_time
Age_stats_playing_time
Born_stats_playing_time
MP_stats_playing_time
Min_stats_playing_time
Mn/MP
Min%
90s_stats_playing_time
Starts_stats_playing_time
Mn/Start
Compl
Subs
Mn/Sub
unSub
PPM
onG
onGA
+/-
+/-90
On-Off
Rk_stats_misc
Nation_stats_misc
Pos_stats_misc
Comp_stats_misc
Age_stats_misc
Born_stats_misc
90s_stats_misc
Cr

### 1.5 Check Required Recruitment Metrics

Check whether the initial dataset contains the metrics required for the Defensive Controller analysis.

The desired features include defensive actions, progressive passing and carrying, possession involvement, ball security and recoveries.

This check showed that several important variables were unavailable, motivating the switch to the richer SofaScore dataset below.

In [7]:
expected_columns = [
    "Tkl",
    "Blocks",
    "Int",
    "Tkl+Int",
    "Clr",
    "Err",
    "PrgP",
    "PrgC",
    "KP",
    "Cmp%_stats_passing",
    "xA",
    "PPA",
    "Touches",
    "Carries",
    "PrgR",
    "Mis",
    "Dis",
    "Recov",
]

for column in expected_columns:
    print(column, column in df.columns)

Tkl False
Blocks False
Int True
Tkl+Int False
Clr False
Err False
PrgP False
PrgC False
KP False
Cmp%_stats_passing False
xA False
PPA False
Touches False
Carries False
PrgR False
Mis False
Dis False
Recov False


## 2. SofaScore 2025/26 Data

The initial dataset did not contain sufficient passing, possession,
progression and defensive metrics for the recruitment analysis.

A richer 2025/26 dataset was therefore collected from SofaScore across
Europe's Big Five leagues.

### 2.1 Define the Raw Data Directory

Set the directory containing the raw 2025/26 SofaScore files.

All five Big Five league datasets are stored in the same directory so they can be discovered and loaded programmatically.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "2025-26"

RAW_DATA_DIR

WindowsPath('c:/Users/Advaith R/Documents/Projects/man-utd-recruitment-analysis/data/raw/2025-26')

### 2.2 Discover SofaScore League Files

Search the raw data directory for SofaScore CSV files corresponding to the 2025/26 season.

Using a filename pattern avoids hard-coding individual league filenames and makes the loading process easier to reproduce.

In [5]:
sofascore_files = sorted(
    RAW_DATA_DIR.glob("sofascore__*__2025_2026.csv")
)

for file in sofascore_files:
    print(file.name)

sofascore__England_Premier_League__2025_2026.csv
sofascore__France_Ligue_1__2025_2026.csv
sofascore__Germany_Bundesliga__2025_2026.csv
sofascore__Italy_Serie_A__2025_2026.csv
sofascore__Spain_La_Liga__2025_2026.csv


### 2.3 Load Each League Dataset

Load each SofaScore league file into a separate DataFrame and store them in a dictionary.

The shape of each dataset is printed to confirm that all five leagues were loaded successfully and to inspect their player coverage.

In [6]:
league_dfs = {}

for file in sofascore_files:
    df_league = pd.read_csv(file)
    league_dfs[file.stem] = df_league

for name, df_league in league_dfs.items():
    print(name, df_league.shape)

sofascore__England_Premier_League__2025_2026 (537, 117)
sofascore__France_Ligue_1__2025_2026 (551, 117)
sofascore__Germany_Bundesliga__2025_2026 (499, 117)
sofascore__Italy_Serie_A__2025_2026 (586, 117)
sofascore__Spain_La_Liga__2025_2026 (600, 117)


### 2.4 Validate Schema Consistency

Before combining the five leagues, verify that every dataset contains exactly the same columns in the same order.

Concatenating datasets with inconsistent schemas could introduce unexpected missing columns or misaligned data.

In [7]:
reference_columns = list(next(iter(league_dfs.values())).columns)

for name, df_league in league_dfs.items():
    same_schema = list(df_league.columns) == reference_columns
    print(name, same_schema)

sofascore__England_Premier_League__2025_2026 True
sofascore__France_Ligue_1__2025_2026 True
sofascore__Germany_Bundesliga__2025_2026 True
sofascore__Italy_Serie_A__2025_2026 True
sofascore__Spain_La_Liga__2025_2026 True


### 2.5 Combine the Big Five Leagues

Concatenate the five league datasets into a single DataFrame representing the 2025/26 Big Five European leagues.

The combined shape and league counts are checked to confirm that all league records were retained.

In [8]:
big5_df = pd.concat(
    league_dfs.values(),
    ignore_index=True
)

big5_df.shape

(2773, 117)

In [9]:
big5_df["league"].value_counts()

league
Spain La Liga             600
Italy Serie A             586
France Ligue 1            551
England Premier League    537
Germany Bundesliga        499
Name: count, dtype: int64

## Data Quality Checks

Before defining the recruitment candidate pool, the combined dataset is
validated for duplicate records, missing identifiers, league coverage and
playing-time consistency.

### 3.1 Check Core Data Quality

Perform basic validation of the combined dataset.

We check:
- total number of records
- number of unique players
- missing player identifiers
- repeated player identifiers
- missing player names
- missing teams
- missing playing-time data

Repeated player IDs are investigated rather than immediately removed because a player may legitimately appear multiple times after representing more than one club during the season.

In [10]:
print("Shape:", big5_df.shape)
print("Unique players:", big5_df["sofascore_id"].nunique())
print("Missing player IDs:", big5_df["sofascore_id"].isna().sum())
print("Duplicate player IDs:", big5_df["sofascore_id"].duplicated().sum())
print("Missing player names:", big5_df["player"].isna().sum())
print("Missing teams:", big5_df["team"].isna().sum())
print("Missing minutes:", big5_df["minutes"].isna().sum())

Shape: (2773, 117)
Unique players: 2691
Missing player IDs: 0
Duplicate player IDs: 82
Missing player names: 0
Missing teams: 0
Missing minutes: 0


### 3.2 Investigate Repeated Player IDs

Inspect players appearing more than once in the combined dataset.

A repeated `sofascore_id` does not necessarily represent a duplicate record. Players who transferred during the season can have separate rows for different clubs.

We therefore also check for duplicate combinations of player ID and team ID. If none exist, the repeated player records can be retained and aggregated later.

In [11]:
duplicate_players = big5_df[
    big5_df["sofascore_id"].duplicated(keep=False)
].sort_values(["sofascore_id", "team"])

duplicate_players[
    ["sofascore_id", "player", "team", "league", "games", "starts", "minutes"]
].head(30)

,sofascore_id,player,team,league,games,starts,minutes
2162,45923,Ciro Immobile,Bologna,Italy Serie A,6,2,191
893,45923,Ciro Immobile,Paris FC,France Ligue 1,12,9,711
1263,48480,Pascal Groß,Borussia Dortmund,Germany Bundesliga,11,7,618
46,48480,Pascal Groß,Brighton & Hove Albion,England Premier League,19,18,1636
2103,132645,Niclas Füllkrug,AC Milan,Italy Serie A,20,3,647
531,132645,Niclas Füllkrug,West Ham United,England Premier League,8,5,408
2497,158615,Matías Vecino,Celta Vigo,Spain La Liga,6,2,143
1924,158615,Matías Vecino,Lazio,Italy Serie A,13,5,605
1048,174975,Arthur Masuaku,RC Lens,France Ligue 1,4,3,229
360,174975,Arthur Masuaku,Sunderland,England Premier League,3,2,127


In [12]:
same_team_duplicates = big5_df.duplicated(
    subset=["sofascore_id", "sofascore_team_id"],
    keep=False
)

print("Same player-team duplicate rows:", same_team_duplicates.sum())

Same player-team duplicate rows: 0


### 3.3 Assess Missing Values

Calculate the number and percentage of missing values for every column.

This helps identify metrics with insufficient coverage and prevents missing observations from being silently converted to zero, which could incorrectly imply that an event did not occur.

In [13]:
missing_summary = (
    big5_df.isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary = pd.DataFrame({
    "missing_count": missing_summary,
    "missing_pct": (missing_summary / len(big5_df) * 100).round(2)
})

missing_summary.head(30)

,missing_count,missing_pct
goals_prevented,2583,93.15
xg,481,17.35
xag,28,1.01
crosses_completed,0,0.00
chipped_passes_completed,0,0.00
long_balls_pct,0,0.00
passes_opp_half,0,0.00
passes_own_half,0,0.00
passes_completed,0,0.00
pass_completion_pct,0,0.00


### 3.4 Validate Metric Distributions

Inspect descriptive statistics for important playing-time, passing, possession and defensive variables.

This acts as a sanity check for unrealistic values and also confirms that most event statistics are stored as season totals rather than per-90 values.

In [14]:
big5_df[
    [
        "games",
        "starts",
        "minutes",
        "passes_total",
        "pass_completion_pct",
        "touches",
        "tackles",
        "interceptions",
        "ball_recoveries",
        "duels_won",
        "possession_lost"
    ]
].describe().round(2)

,games,starts,minutes,passes_total,pass_completion_pct,touches,tackles,interceptions,ball_recoveries,duels_won,possession_lost
count,2773.00,2773.00,2773.00,2773.0,2773.00,2773.00,2773.00,2773.00,2773.00,2773.00,2773.00
mean,19.59,13.89,1253.17,558.5,79.48,798.09,20.23,10.46,59.17,60.68,158.66
std,11.49,11.24,964.41,527.4,13.19,685.36,20.36,11.97,58.94,53.51,132.84
min,1.00,0.00,1.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,9.00,3.00,357.00,113.0,75.08,186.00,3.00,1.00,14.00,13.00,44.00
50%,21.00,12.00,1128.00,413.0,82.14,652.00,14.00,6.00,45.00,49.00,134.00
75%,30.00,23.00,2006.00,869.0,87.10,1268.00,31.00,17.00,86.00,97.00,243.00
max,38.00,38.00,3420.00,3035.0,100.00,3408.00,120.00,78.00,399.00,297.00,665.00


#### Findings

The distributions do not indicate any immediately implausible values in the core metrics.

- Maximum playing time is **3,420 minutes**, equivalent to 38 full 90-minute matches, which is consistent with a complete league season.
- Passing, touches, tackles, interceptions and recoveries show wide distributions, but their maximum values remain plausible as season totals.
- Pass completion percentages also fall within a credible range.
- The large differences in event totals are expected because players have accumulated very different amounts of playing time.

The statistics therefore appear suitable for further preparation. However, raw event totals should not be compared directly between players because they are influenced by minutes played. Relevant count statistics will later be converted to **per-90 rates** after applying a minimum playing-time threshold.

## 4. Player Metadata Enrichment

### 4.1 Establish the Metadata Collection Population

The season-level SofaScore dataset contains detailed performance statistics but does not provide sufficiently detailed positional information for defining the Defensive Controller candidate pool.

Additional player metadata is therefore required.

Before using the metadata, confirming the number of records and unique players in the complete Big Five dataset. The distinction matters because players who changed clubs during the season can appear in multiple rows.

In [15]:
print("Rows:", len(big5_df))
print("Unique SofaScore players:", big5_df["sofascore_id"].nunique())

Rows: 2773
Unique SofaScore players: 2691


#### Findings

The combined dataset contains **2,773 player-team records representing 2,691 unique players**.

The difference of 82 records is consistent with the earlier finding that some players represented more than one club during the season.

Metadata therefore needs to be collected and matched using the unique `sofascore_id`, rather than assuming that every row represents a different player.

### 4.2 Explore Playing-Time Thresholds for Metadata Collection

Collecting additional metadata for every player would require unnecessary requests for players with very limited involvement.

To reduce this workload, explore how many unique players remain at several playing-time thresholds.

This is an **enrichment threshold**, not the final modelling threshold. Its purpose is to retain a broad pool of potentially relevant players while reducing unnecessary metadata collection.

In [16]:
for threshold in [500, 900, 1200, 1500, 1800, 2000]:
    players = big5_df.loc[
        big5_df["minutes"] >= threshold,
        "sofascore_id"
    ].nunique()

    print(f"{threshold:4} minutes: {players} players")
    

 500 minutes: 1932 players
 900 minutes: 1591 players
1200 minutes: 1329 players
1500 minutes: 1101 players
1800 minutes: 868 players
2000 minutes: 696 players


#### Findings

The number of players falls substantially as the threshold increases:

- 500 minutes → 1,932 players
- 900 minutes → 1,591 players
- 1,200 minutes → 1,329 players
- 1,500 minutes → 1,101 players
- 1,800 minutes → 868 players
- 2,000 minutes → 696 players

A **900-minute threshold** provides a useful compromise for metadata enrichment: it removes players with very limited participation while retaining a broad population from which the eventual recruitment pool can be constructed.

A stricter threshold will be considered later for the actual performance analysis.

### 4.3 Calculate Player-Level Season Minutes

The previous exploration operates on individual player-team rows. This can understate the playing time of players who transferred during the season because their minutes are divided across multiple clubs.

To avoid excluding these players incorrectly, aggregate minutes by player across all clubs represented during 2025/26.

The number of clubs represented is also retained so that transfer cases can be identified.

In [17]:
player_minutes = (
    big5_df
    .groupby(["sofascore_id", "player"], as_index=False)
    .agg(
        season_minutes=("minutes", "sum"),
        clubs=("team", "nunique")
    )
)

player_minutes.head()

,sofascore_id,player,season_minutes,clubs
0,791,James Milner,802,1
1,2899,Daniele Padelli,180,1
2,3041,Raúl Albiol,502,1
3,3577,Dante,852,1
4,8959,Manuel Neuer,1860,1


In [18]:
metadata_candidates = player_minutes[
    player_minutes["season_minutes"] >= 900
].copy()

print("Players to enrich:", len(metadata_candidates))
print("Multi-club players:", (metadata_candidates["clubs"] > 1).sum())

Players to enrich: 1602
Multi-club players: 52


#### Findings

Using **combined season minutes** results in **1,602 players** meeting the 900-minute metadata threshold, slightly more than the row-level calculation.

Among them, **52 players represented multiple clubs** during the season.

This confirms why aggregation should occur before applying season-level eligibility rules: evaluating individual club rows could incorrectly exclude transfer players whose total seasonal involvement is substantial.

### 4.4 Load Player Metadata

Load the separately collected player-profile metadata.

This enriches the performance dataset with information not available in the season-level statistics, including:

- broad and detailed playing position
- date of birth
- height and weight
- preferred foot
- nationality
- estimated market value

The metadata was collected separately using each player's unique SofaScore identifier.

In [19]:
metadata_df = pd.read_csv(
    RAW_DATA_DIR / "sofascore_player_metadata.csv"
)

print(metadata_df.shape)
metadata_df.head()

(1372, 13)


,sofascore_id,player,current_team,position,positions_detailed,date_of_birth,height,weight,preferred_foot,country,market_value,market_value_currency,season_minutes
0,8959,Manuel Neuer,FC Bayern München,G,['GK'],1986-03-27,193,NaN,Right,Germany,4099999.0,EUR,1860
1,15219,Lukáš Hrádecký,AS Monaco,G,['GK'],1989-11-24,190,80.0,Right,Finland,1600000.0,EUR,1441
2,15466,Luka Modrić,AC Milan,M,"['MC', 'DM']",1985-09-09,173,NaN,Right,Croatia,3800000.0,EUR,2821
3,16206,Yann Sommer,Club Brugge KV,G,['GK'],1988-12-17,183,79.0,Right,Switzerland,1800000.0,EUR,2961
4,17651,Santi Cazorla,No team,M,"['AM', 'MC', 'DM']",1984-12-13,168,66.0,Both,Spain,NaN,NaN,940


#### Findings and Data Limitation

Metadata was successfully collected for **1,372 players**, compared with **1,602 players** identified for enrichment.

This represents approximately **85.6% coverage** of the intended metadata population.

Further automated requests were stopped after the source began returning HTTP 403 challenge responses rather than attempting to circumvent the source's access controls.

As a result, the subsequent positional candidate pool is limited to players for whom metadata was successfully collected. The final shortlist should therefore be interpreted as the strongest candidates **among the observed eligible population**, rather than as an exhaustive ranking of every eligible midfielder in Europe's Big Five leagues.

### 4.5 Inspect Positional Metadata

Examine the broad SofaScore position categories and the available detailed positional combinations.

The broad position variable identifies general roles such as midfielder (`M`), defender (`D`), forward (`F`) and goalkeeper (`G`).

For this recruitment problem, however, the detailed positions are more useful because they allow central midfield (`MC`) and defensive midfield (`DM`) players to be identified specifically.

In [20]:
print(metadata_df["position"].value_counts(dropna=False))

print(
    metadata_df["positions_detailed"]
    .value_counts(dropna=False)
    .head(20)
)

position
M    535
D    467
F    261
G    109
Name: count, dtype: int64
positions_detailed
['MC', 'DM']          200
['DC']                154
['ST']                142
['GK']                109
['DR']                 58
['DC', 'DR']           52
['DL']                 52
['LW']                 40
['DC', 'DL']           37
['RW']                 27
['DL', 'ML']           27
['AM', 'MC']           26
['MC']                 25
['MC', 'AM']           25
['DR', 'MR']           23
['DR', 'DC']           20
['MR', 'DR']           20
['MC', 'DM', 'AM']     19
['AM']                 18
['ST', 'AM']           16
Name: count, dtype: int64


#### Findings

Of the **1,372 players with collected metadata**, 535 are classified broadly as midfielders.

The detailed position field contains combinations such as `['MC', 'DM']`, allowing the analysis to distinguish central and defensive midfield profiles from attacking or wide midfielders.

This makes the detailed position information substantially more useful than the broad `M` classification for defining the Defensive Controller candidate pool.

### 4.6 Explore Detailed Midfielder Positions

Restrict the metadata temporarily to players classified broadly as midfielders and inspect the frequency of their detailed positional combinations.

This helps determine which position codes should define the initial recruitment universe before performance characteristics are considered.

In [21]:
midfielders = metadata_df[
    metadata_df["position"] == "M"
].copy()

position_profiles = (
    midfielders["positions_detailed"]
    .value_counts()
    .reset_index()
)

position_profiles.columns = [
    "positions_detailed",
    "players"
]

position_profiles

,positions_detailed,players
0,"['MC', 'DM']",199
1,['MC'],25
2,"['AM', 'MC']",24
3,"['MC', 'AM']",24
4,['LW'],22
...,...,...
70,"['LW', 'ST']",1
71,"['DR', 'RW', 'MR']",1
72,"['MC', 'RW', 'LW']",1
73,"['MC', 'DC', 'DM']",1


### 4.7 Parse Detailed Position Lists

The detailed positions were loaded from CSV as strings such as `"['MC', 'DM']"` rather than actual Python lists.

Converting these strings into list objects using `ast.literal_eval`. This allows membership tests such as `"DM" in positions` to be performed safely when constructing the positional candidate pool.

In [22]:
import ast

midfielders["positions_list"] = (
    midfielders["positions_detailed"]
    .apply(ast.literal_eval)
)


### 4.8 Define the Defensive Controller Positional Pool

Retaining midfielders whose detailed positional profile contains either:

- `DM` — Defensive Midfielder
- `MC` — Central Midfielder

The condition deliberately uses **DM OR MC**, rather than requiring a player to be classified in both positions.

Position is being used as an initial eligibility criterion rather than as evidence of role suitability. The subsequent performance model will determine whether each eligible midfielder actually exhibits the defensive, possession and progression characteristics required for the Defensive Controller role.

In [23]:
controller_pool = midfielders[
    midfielders["positions_list"].apply(
        lambda positions:
        "DM" in positions or "MC" in positions
    )
].copy()

print("All midfielders:", len(midfielders))
print("Controller positional pool:", len(controller_pool))

All midfielders: 535
Controller positional pool: 349


#### Findings

The metadata contains **535 midfielders**, of whom **349** are classified as either central midfielders or defensive midfielders.

This reduces the population substantially before any performance scoring is applied and removes midfield profiles that are less relevant to the role, such as players operating exclusively in attacking or wide areas.

Importantly, these 349 players have not yet been judged as good Defensive Controllers; they have only passed the positional eligibility criterion.

In [24]:
controller_pool[
    "positions_detailed"
].value_counts()

positions_detailed
['MC', 'DM']                199
['MC']                       25
['AM', 'MC']                 24
['MC', 'AM']                 24
['MC', 'DM', 'AM']           19
['DM', 'MC']                 13
['MC', 'AM', 'DM']            7
['MC', 'DM', 'DC']            7
['AM', 'MC', 'DM']            6
['MC', 'ML']                  3
['MC', 'MR']                  2
['MR', 'MC']                  2
['AM', 'MC', 'RW']            2
['DC', 'MC', 'DM']            1
['DC', 'DM', 'MC']            1
['MC', 'DM', 'LW']            1
['MC', 'LW']                  1
['LW', 'MC', 'DM']            1
['MC', 'DM', 'AM', 'RW']      1
['MC', 'DM', 'DR']            1
['AM', 'MC', 'MR']            1
['MC', 'RW', 'DM', 'AM']      1
['RW', 'MC']                  1
['ML', 'MC']                  1
['AM', 'MC', 'LW']            1
['LW', 'MC']                  1
['MC', 'RW', 'LW']            1
['MC', 'DC', 'DM']            1
['MC', 'DM', 'DL']            1
Name: count, dtype: int64

## 5. Connect Positional Eligibility to Performance Data

### 5.1 Match Eligible Midfielders to Season Statistics

Use the unique SofaScore player IDs from the positional pool to retrieve the corresponding records from the complete Big Five performance dataset.

Because the performance dataset remains at player-team level, players who changed clubs may still appear more than once at this stage.

In [25]:
controller_ids = set(
    controller_pool["sofascore_id"].astype(int)
)

In [26]:
controller_stats = big5_df[
    big5_df["sofascore_id"]
    .astype(int)
    .isin(controller_ids)
].copy()

print("Rows:", len(controller_stats))
print(
    "Unique players:",
    controller_stats["sofascore_id"].nunique()
)

Rows: 359
Unique players: 349


#### Findings

The positional pool produces **359 player-team records representing 349 unique players**.

The additional 10 rows indicate that some eligible midfielders represented multiple clubs during the season.

These records should not simply be removed as duplicates because each contains genuine performance accumulated at a different club. They must instead be combined into complete player-season records.

### 5.2 Identify Multi-Club Players

Identifying players who have more than one club-level record and calculate their combined minutes.

This validates the reason for the difference between player count and row count and prepares the dataset for season-level aggregation.

In [27]:
transfer_check = (
    controller_stats
    .groupby(["sofascore_id", "player"])
    .agg(
        clubs=("team", "nunique"),
        rows=("team", "size"),
        total_minutes=("minutes", "sum")
    )
    .reset_index()
)

print(
    "Players:",
    len(transfer_check)
)

print(
    "Multi-club players:",
    (transfer_check["clubs"] > 1).sum()
)

transfer_check[
    transfer_check["clubs"] > 1
].sort_values(
    "total_minutes",
    ascending=False
)

Players: 349
Multi-club players: 10


,sofascore_id,player,clubs,rows,total_minutes
50,250737,Adrien Rabiot,2,2,2640
239,929620,Fabian Rieder,2,2,2498
7,48480,Pascal Groß,2,2,2254
194,894733,Marshall Munetsi,2,2,1868
211,904970,Conor Gallagher,2,2,1861
293,997183,Xavi Simons,2,2,1859
344,1127590,Mohamed Kaba,2,2,1570
93,609752,Guido Rodríguez,2,2,1485
174,867441,Angel Gomes,2,2,1380
191,891829,Matt O'Riley,2,2,942


#### Findings

The positional pool contains **349 unique players**, including **10 players who represented multiple clubs** during 2025/26.

Their club-level records will therefore be aggregated before any per-90 calculations or role scoring.

This ensures that a transferred player's evaluation reflects their complete league season rather than treating each spell as a separate candidate.

## 6. Create Player-Season Performance Records

### 6.1 Define Additive Performance Metrics

To combine the records of players who changed clubs, first identify statistics that can legitimately be summed across teams.

These are primarily event counts such as minutes, passes, tackles, interceptions, recoveries and duels.

Percentage metrics are deliberately excluded because percentages should not be added across club spells. They will instead be recalculated later from the aggregated numerators and denominators.

In [28]:
additive_metrics = [
    # Playing time
    "games",
    "starts",
    "minutes",

    # Passing
    "passes_total",
    "passes_completed",
    "passes_own_half_total",
    "passes_own_half",
    "passes_opp_half_total",
    "passes_opp_half",
    "passes_final_third",
    "long_balls_total",
    "long_balls_completed",

    # Possession
    "touches",
    "possession_lost",
    "dispossessed",

    # Defensive
    "tackles",
    "tackles_won",
    "interceptions",
    "outfield_blocks",
    "clearances",
    "ball_recoveries",
    "possession_won_att_third",
    "dribbled_past",

    # Duels
    "duels_won",
    "duels_lost",
    "ground_duels_won",
    "aerials_won",
    "aerials_lost",

    # Carrying / dribbling
    "dribbles_attempted",
    "dribbles_completed",

    # Discipline
    "fouls",
    "fouled",
    "yellow_cards",
    "red_cards",

    # Creativity
    "key_passes",
    "attempt_assists",
    "big_chances_created"
]

### 6.2 Validate Required Aggregation Columns

Check that every metric selected for season-level aggregation is present in the SofaScore dataset.

This prevents the aggregation pipeline from silently omitting an intended feature because of a missing or incorrectly named column.

In [29]:
missing_columns = [
    col for col in additive_metrics
    if col not in controller_stats.columns
]

print("Missing columns:", missing_columns)

Missing columns: []


#### Findings

No required aggregation columns are missing.

The complete set of selected passing, possession, defensive, duel and discipline statistics is therefore available for all subsequent player-season transformations.

### 6.3 Aggregate Performance to Player-Season Level

Grouping the eligible midfielder data by unique player and sum all additive statistics.

This converts the player-team dataset into one record per player for the entire 2025/26 league season.

Aggregation is performed **before** calculating per-90 statistics so that transferred players are evaluated using their complete season totals.

In [30]:
controller_season = (
    controller_stats
    .groupby(
        ["sofascore_id", "player"],
        as_index=False
    )[additive_metrics]
    .sum()
)

print(controller_season.shape)
print(
    "Unique players:",
    controller_season["sofascore_id"].nunique()
)

(349, 39)
Unique players: 349


#### Findings

The aggregation produces **349 rows and 349 unique player IDs**, confirming that the dataset now contains exactly one performance record per eligible midfielder.

The 10 multi-club cases have therefore been successfully consolidated without removing their performance data.


### 6.4 Preserve Club and League Context

Aggregating player statistics to season level removes the original team and league information.

However, this information remains important for recruitment analysis, particularly for players who represented multiple clubs during the season.

We therefore create a separate player-level dataset containing all clubs and leagues represented by each player.

Multiple club or league names are joined using `|`, preserving transfer history while maintaining one record per player.


In [31]:
player_context = (
    controller_stats
    .groupby(
        ["sofascore_id", "player"],
        as_index=False
    )
    .agg(
        season_teams=(
            "team",
            lambda x: " | ".join(sorted(set(x)))
        ),
        season_leagues=(
            "league",
            lambda x: " | ".join(sorted(set(x)))
        )
    )
)

In [32]:
controller_season = controller_season.merge(
    player_context,
    on=["sofascore_id", "player"],
    how="left"
)


#### Findings

The season-level performance dataset now retains the club and league history of each player.

For players who represented multiple clubs, their performance statistics remain aggregated into one season record, while `season_teams` preserves the names of the clubs involved.

This allows the analysis to retain complete season performance without losing the contextual information required for later recruitment eligibility decisions.



### 6.5 Recalculate Percentage Statistics

Following player-season aggregation, percentage statistics must be recalculated using the combined event totals.

Unlike event counts, percentages cannot be summed or simply averaged across clubs because players may have recorded different numbers of attempts at each club.

For example, a player completing 90 of 100 passes at one club and 40 of 50 passes at another has a season pass completion rate of:

**(90 + 40) / (100 + 50) × 100 = 86.7%**

Simply averaging the two club percentages would produce 85%, which does not accurately represent the player's combined performance.

We therefore recalculate four metrics:

- **Pass Completion %:** Completed passes / Total passes
- **Long Ball Completion %:** Completed long balls / Total long balls
- **Dribble Success %:** Completed dribbles / Attempted dribbles
- **Duel Win %:** Duels won / Total duels contested

This ensures that each player's percentage statistics reflect their complete 2025/26 season.


In [33]:
controller_season["pass_completion_pct"] = (
    controller_season["passes_completed"]
    / controller_season["passes_total"]
    * 100
)

In [34]:
controller_season["long_balls_pct"] = (
    controller_season["long_balls_completed"]
    / controller_season["long_balls_total"]
    * 100
)

In [35]:
controller_season["dribbles_pct"] = (
    controller_season["dribbles_completed"]
    / controller_season["dribbles_attempted"]
    * 100
)

In [36]:
controller_season["duels_won_pct"] = (
    controller_season["duels_won"]
    / (
        controller_season["duels_won"]
        + controller_season["duels_lost"]
    )
    * 100
)


### 6.6 Validate Recalculated Percentage Metrics

Check that the recalculated percentage metrics fall within the mathematically valid range of 0–100%.

We also inspect missing values, which may occur when a player has no recorded attempts for a particular event.

This validation helps detect incorrect aggregation, inconsistent numerators and denominators, or unexpected division results before the data is used for modelling.


In [105]:
percentage_columns = [
    "pass_completion_pct",
    "long_balls_pct",
    "dribbles_pct",
    "duels_won_pct"
]

percentage_validation = pd.DataFrame({
    "min": controller_season[percentage_columns].min(),
    "max": controller_season[percentage_columns].max(),
    "missing": controller_season[percentage_columns].isna().sum(),
    "out_of_range": (
        (controller_season[percentage_columns] < 0)
        | (controller_season[percentage_columns] > 100)
    ).sum()
})

percentage_validation.round(2)

,min,max,missing,out_of_range
pass_completion_pct,63.79,94.21,0,0
long_balls_pct,26.67,81.82,0,0
dribbles_pct,0.00,100.00,0,0
duels_won_pct,33.88,68.00,0,0



#### Findings: Percentage Metric Validation

All four recalculated percentage metrics passed the validation checks across the **349 player-season records**.

| Metric | Minimum | Maximum | Missing Values | Out-of-Range Values |
|---|---:|---:|---:|---:|
| Pass Completion % | 63.79% | 94.21% | 0 | 0 |
| Long Ball Completion % | 26.67% | 81.82% | 0 | 0 |
| Dribble Success % | 0.00% | 100.00% | 0 | 0 |
| Duel Win % | 33.88% | 68.00% | 0 | 0 |

**Key observations:**

- All calculated percentages fall within the mathematically valid range of 0–100%.
- No missing values were identified in any of the four metrics.
- Passing and duel success rates fall within plausible ranges for professional footballers.
- Dribble success rates span the full 0–100% range. These extremes are mathematically valid, although they may reflect players with relatively few dribble attempts.

**Conclusion:** The recalculated percentage metrics pass the initial range and completeness checks and are suitable for subsequent analysis. This does not independently verify the accuracy of the underlying event data.

The next stage will combine these validated season-level performance records with player metadata.



### 6.7 Validate Player-Season Aggregation

Verify that the aggregated dataset contains exactly one record per player and that the original playing-time information has been preserved.

The validation checks:

- Number of player-season records and unique player IDs.
- Duplicate player IDs after aggregation.
- Total minutes before and after aggregation.
- Missing club or league information.

Preserving total minutes is particularly important because players who transferred during the season had their club-level records combined.

Successful validation would confirm that the aggregation has not accidentally duplicated or discarded playing-time records.


In [106]:
print("Player-season rows:", len(controller_season))

print(
    "Unique player IDs:",
    controller_season["sofascore_id"].nunique()
)

print(
    "Duplicate player IDs:",
    controller_season["sofascore_id"].duplicated().sum()
)

print(
    "Original total minutes:",
    controller_stats["minutes"].sum()
)

print(
    "Aggregated total minutes:",
    controller_season["minutes"].sum()
)

print(
    "Missing team information:",
    controller_season["season_teams"].isna().sum()
)

print(
    "Missing league information:",
    controller_season["season_leagues"].isna().sum()
)

Player-season rows: 349
Unique player IDs: 349
Duplicate player IDs: 0
Original total minutes: 685819
Aggregated total minutes: 685819
Missing team information: 0
Missing league information: 0



#### Findings: Player-Season Aggregation Validation

The player-season aggregation passed all structural and playing-time preservation checks.

| Validation Check | Result |
|---|---:|
| Player-season records | 349 |
| Unique player IDs | 349 |
| Duplicate player IDs | 0 |
| Original total minutes | 685,819 |
| Aggregated total minutes | 685,819 |
| Missing team information | 0 |
| Missing league information | 0 |

**Key findings:**

- All 349 eligible midfielders have exactly one player-season record.
- The total playing time remains unchanged at **685,819 minutes**, confirming that no minutes were lost or duplicated during aggregation.
- No duplicate player IDs remain.
- Team and league information has been preserved for every player.

**Conclusion:** The aggregation successfully converted player-team records into player-season records while preserving playing time and recruitment context.

The dataset is now ready for metadata integration and further feature engineering.


In [37]:
print("Shape:", controller_season.shape)
print(
    "Unique players:",
    controller_season["sofascore_id"].nunique()
)

controller_season[
    [
        "player",
        "season_teams",
        "minutes",
        "passes_total",
        "pass_completion_pct",
        "touches",
        "tackles",
        "interceptions",
        "ball_recoveries"
    ]
].head(10)

Shape: (349, 45)
Unique players: 349


,player,season_teams,minutes,passes_total,pass_completion_pct,touches,tackles,interceptions,ball_recoveries
0,Luka Modrić,AC Milan,2821,2219,90.401082,2678,51,39,188
1,Santi Cazorla,Real Oviedo,940,543,86.556169,761,17,4,59
2,Axel Witsel,Girona FC,2533,1466,91.200546,1816,37,30,167
3,Henrikh Mkhitaryan,Inter,1555,997,88.064193,1312,33,14,100
4,Dani Parejo,Villarreal,1398,901,88.013319,1123,42,12,68
5,Jordan Henderson,Brentford,1929,1158,77.892919,1382,27,17,98
6,Nemanja Matić,Sassuolo,2669,1721,86.461360,2064,43,28,165
7,Pascal Groß,Borussia Dortmund | Brighton & Hove Albion,2254,1537,86.337020,1971,37,18,106
8,Benjamin André,Lille,2333,1429,83.414976,1871,66,33,143
9,Nemanja Gudelj,Sevilla,2370,1106,88.336347,1455,35,25,80



## 7. Metadata Integration and Age Calculation

### 7.1 Select Recruitment Metadata

The aggregated dataset contains performance statistics and season-level club information, but it does not yet include several important recruitment attributes.

Select the relevant fields from the separately collected SofaScore player metadata:

- **Position:** Broad and detailed positional classification.
- **Date of birth:** Used to calculate age at the end of the 2025/26 season.
- **Physical attributes:** Height and weight.
- **Preferred foot and nationality:** Additional scouting context.
- **Estimated market value:** Used later to evaluate performance relative to valuation.

The `sofascore_id` is retained as the unique identifier used to merge metadata with performance records.

The metadata's `current_team` field is not used to replace `season_teams`, because current club information may differ from the clubs represented during the 2025/26 season.


In [38]:
metadata_cols = [
    "sofascore_id",
    "position",
    "positions_detailed",
    "date_of_birth",
    "height",
    "preferred_foot",
    "country",
    "current_team",
    "market_value",
    "market_value_currency"
]

controller_metadata = metadata_df[
    metadata_cols
].copy()


### 7.2 Merge Player Metadata

Merge the selected metadata into the aggregated player-season performance dataset using `sofascore_id`.

A left join preserves all 349 player-season records while adding their available recruitment attributes.

Both identifier columns are converted to the same integer data type before merging to prevent mismatches caused by inconsistent data types.

The merge is validated to ensure that it does not introduce duplicate player records or unexpectedly missing metadata.


In [39]:
controller_season["sofascore_id"] = (
    controller_season["sofascore_id"].astype(int)
)

controller_metadata["sofascore_id"] = (
    controller_metadata["sofascore_id"].astype(int)
)

In [40]:
controller_season = controller_season.merge(
    controller_metadata,
    on="sofascore_id",
    how="left",
    validate="one_to_one"
)


### 7.3 Validate Metadata Integration

Verify that the metadata merge preserved the original player population and successfully populated the required recruitment attributes.

Check the total number of records, uniqueness of player identifiers and missing values in position, date of birth and estimated market value.

This ensures that the metadata can be used reliably for age-based eligibility filtering and subsequent recruitment analysis.


In [107]:
print("Rows:", len(controller_season))

print(
    "Unique players:",
    controller_season["sofascore_id"].nunique()
)

print(
    "Duplicate player IDs:",
    controller_season["sofascore_id"].duplicated().sum()
)

print(
    "Missing positions:",
    controller_season["position"].isna().sum()
)

print(
    "Missing dates of birth:",
    controller_season["date_of_birth"].isna().sum()
)

print(
    "Missing market values:",
    controller_season["market_value"].isna().sum()
)

Rows: 349
Unique players: 349
Duplicate player IDs: 0
Missing positions: 0
Missing dates of birth: 0
Missing market values: 3



#### Findings: Metadata Integration

The metadata merge successfully preserved all 349 player-season records.

| Validation Check | Result |
|---|---:|
| Player-season records | 349 |
| Unique player IDs | 349 |
| Duplicate player IDs | 0 |
| Missing positions | 0 |
| Missing dates of birth | 0 |
| Missing estimated market values | 3 |

**Key findings:**

- All 349 players retained their unique identifiers after the merge.
- Positional information and dates of birth are complete, allowing subsequent positional validation and age calculations.
- Three players (0.86%) have missing estimated market values.

**Decision:** Players with missing market values will be retained for performance analysis because market valuation is not an input to the Defensive Controller Score.

Missing valuations will remain as `NaN` rather than being replaced with zero, since a missing valuation does not imply that a player has no market value.

The dataset is ready for age calculation and performance eligibility filtering.



### 7.4 Calculate Player Age

Calculate each player's age using a fixed reference date of **30 June 2026**, representing the end of the 2025/26 season.

A fixed reference date ensures that player ages remain consistent whenever the analysis is reproduced.

Date-of-birth values are first converted to pandas datetime objects.

Age is then calculated as the difference between the reference date and the player's date of birth, expressed in years using 365.25 days per year to approximately account for leap years.

Ages are rounded to one decimal place to preserve useful precision when comparing recruitment candidates.

Age is not included in the performance-based Controller Score. It will instead be used later as a separate recruitment eligibility criterion.


In [42]:
controller_season["date_of_birth"] = pd.to_datetime(
    controller_season["date_of_birth"]
)

REFERENCE_DATE = pd.Timestamp("2026-06-30")

controller_season["age"] = (
    REFERENCE_DATE
    - controller_season["date_of_birth"]
).dt.days / 365.25

controller_season["age"] = (
    controller_season["age"]
    .round(1)
)


### 7.5 Validate Calculated Ages

Inspect the distribution of calculated player ages and check for missing or implausible values.

This validation helps confirm that date-of-birth parsing and age calculations were performed correctly.

The objective is to identify unexpected negative ages, unusually high ages or missing values before age-based recruitment eligibility criteria are applied.


In [108]:
print(
    controller_season["age"]
    .describe()
    .round(1)
)

print(
    "\nMissing ages:",
    controller_season["age"].isna().sum()
)

print(
    "Negative ages:",
    (controller_season["age"] < 0).sum()
)

print(
    "Players older than 45:",
    (controller_season["age"] > 45).sum()
)

count    349.0
mean      27.9
std        3.6
min       20.8
25%       25.2
50%       27.8
75%       30.2
max       41.5
Name: age, dtype: float64

Missing ages: 0
Negative ages: 0
Players older than 45: 0



#### Findings: Player Age Validation

Player ages were calculated using 30 June 2026 as the reference date.

| Statistic | Age (Years) |
|---|---:|
| Players | 349 |
| Mean | 27.9 |
| Standard deviation | 3.6 |
| Minimum | 20.8 |
| 25th percentile | 25.2 |
| Median | 27.8 |
| 75th percentile | 30.2 |
| Maximum | 41.5 |

**Key findings:**

- The average midfielder is 27.9 years old, with a median age of 27.8.
- The middle 50% of players are aged between 25.2 and 30.2 years.
- No missing, negative or implausibly high ages were identified.
- The oldest player is 41.5 years old, which is unusual but not inherently invalid.

**Conclusion:** Age calculation passed the initial validation checks.

Age will be retained as a recruitment feasibility criterion rather than incorporated into the performance-based Controller Score. This separates footballing suitability from long-term recruitment considerations.



## 8. Playing-Time Eligibility and Per-90 Standardisation

### 8.1 Apply Minimum Playing-Time Eligibility

The initial dataset includes midfielders with different amounts of playing time during the 2025/26 season.

Players with very limited minutes may exhibit unusually high or low performance rates because their statistics are based on small samples.

To reduce this risk, we apply a minimum threshold of **1,500 season minutes**.

This threshold is applied after aggregating player statistics across clubs, ensuring that transferred players are evaluated using their combined season playing time.

The threshold is a modelling decision intended to improve comparability, rather than a guarantee of statistical reliability.

Players below 1,500 minutes are excluded from the main recruitment analysis.


In [109]:
MIN_MINUTES = 1500

eligible_players = controller_season.loc[
    controller_season["minutes"] >= MIN_MINUTES
].copy()

print("Players before filtering:", len(controller_season))
print("Players after filtering:", len(eligible_players))
print(
    "Players excluded:",
    len(controller_season) - len(eligible_players)
)

print(
    "\nMinimum eligible minutes:",
    eligible_players["minutes"].min()
)

Players before filtering: 349
Players after filtering: 248
Players excluded: 101

Minimum eligible minutes: 1505



### 8.2 Validate Playing-Time Eligibility

Confirm that every retained player satisfies the 1,500-minute threshold.

Inspect the playing-time distribution of eligible players and verify that no duplicate player IDs have been introduced.

This provides a final eligibility check before converting event-count statistics into per-90-minute performance rates.


In [110]:
print(
    eligible_players["minutes"]
    .describe()
    .round(1)
)

print(
    "\nPlayers below threshold:",
    (eligible_players["minutes"] < MIN_MINUTES).sum()
)

print(
    "Duplicate player IDs:",
    eligible_players["sofascore_id"].duplicated().sum()
)

count     248.0
mean     2274.7
std       479.3
min      1505.0
25%      1864.2
50%      2231.5
75%      2675.0
max      3414.0
Name: minutes, dtype: float64

Players below threshold: 0
Duplicate player IDs: 0



#### Findings: Minimum Playing-Time Eligibility

Applying the 1,500-minute threshold reduced the player population from **349 to 248 midfielders**.

| Validation Check | Result |
|---|---:|
| Players before filtering | 349 |
| Players retained | 248 |
| Players excluded | 101 |
| Retention rate | 71.1% |
| Players below threshold | 0 |
| Duplicate player IDs | 0 |

**Playing-time distribution of eligible players:**

| Statistic | Minutes |
|---|---:|
| Mean | 2,274.7 |
| Standard deviation | 479.3 |
| Minimum | 1,505 |
| 25th percentile | 1,864.2 |
| Median | 2,231.5 |
| 75th percentile | 2,675 |
| Maximum | 3,414 |

**Key findings:**

- The threshold excluded 101 players (28.9%) with fewer than 1,500 minutes.
- All 248 retained midfielders satisfy the minimum playing-time requirement.
- No duplicate player IDs were identified.
- The median eligible player accumulated approximately 2,232 minutes during the season.
- The retained players still have different amounts of playing time, making further standardisation necessary.

**Conclusion:** The playing-time filter successfully established a more consistent sample for performance comparisons. However, raw event totals remain influenced by playing time, so selected statistics must be converted into per-90-minute rates.


In [43]:
controller_season[
    [
        "player",
        "age",
        "positions_detailed",
        "season_teams",
        "minutes"
    ]
].sort_values("age").head(10)

,player,age,positions_detailed,season_teams,minutes
348,Jobe Bellingham,20.8,"['MC', 'DM']",Borussia Dortmund,1800
347,Tom Bischof,21.0,"['MC', 'DM']",FC Bayern München,1320
330,Arda Güler,21.3,"['AM', 'MC', 'RW']",Real Madrid,2026
343,Valentín Barco,21.9,"['MC', 'DM', 'DL']",RC Strasbourg,2170
339,Cher Ndour,21.9,"['MC', 'DM']",Fiorentina,1885
334,Andrey Santos,22.2,"['MC', 'DM']",Chelsea,1280
310,Gvidas Gineitis,22.2,"['MC', 'DM']",Torino,1763
311,Yehor Yarmoliuk,22.3,"['MC', 'DM']",Brentford,2675
315,Lesley Ugochukwu,22.3,"['MC', 'DM']",Burnley,2359
335,Adam Wharton,22.4,"['MC', 'DM']",Crystal Palace,2562


### 8.3 Convert Event Statistics to Per-90 Rates

Convert additive event counts into rates per 90 minutes to compare players with different playing time. Each rate is calculated as `(total events / total minutes) * 90`. Completion percentages are kept separately rather than converted to per-90 rates. This adjusts for minutes played, but not tactical context or team possession.


In [46]:
per90_metrics = [
    # Passing / control
    "passes_total",
    "passes_completed",
    "passes_final_third",
    "passes_opp_half",
    "long_balls_completed",

    # Possession
    "touches",
    "possession_lost",
    "dispossessed",

    # Defensive
    "tackles",
    "tackles_won",
    "interceptions",
    "outfield_blocks",
    "ball_recoveries",
    "possession_won_att_third",
    "dribbled_past",

    # Duels
    "duels_won",

    # Carrying
    "dribbles_attempted",
    "dribbles_completed",

    # Discipline
    "fouls",
    "yellow_cards",

    # Creativity
    "key_passes"
]

In [47]:
for metric in per90_metrics:
    eligible_players[f"{metric}_per90"] = (
        eligible_players[metric]
        / eligible_players["minutes"]
        * 90
    )

In [48]:
eligible_players[
    [
        "player",
        "age",
        "minutes",
        "passes_total_per90",
        "pass_completion_pct",
        "passes_final_third_per90",
        "touches_per90",
        "tackles_per90",
        "interceptions_per90",
        "ball_recoveries_per90",
        "duels_won_per90",
        "possession_lost_per90"
    ]
].describe().round(2)

,age,minutes,passes_total_per90,pass_completion_pct,passes_final_third_per90,touches_per90,tackles_per90,interceptions_per90,ball_recoveries_per90,duels_won_per90,possession_lost_per90
count,248.00,248.00,248.00,248.00,248.00,248.00,248.00,248.00,248.00,248.00,248.00
mean,28.12,2274.67,47.35,84.28,10.89,63.25,1.96,0.95,4.67,4.80,11.30
std,3.57,479.30,14.64,5.25,4.74,14.30,0.63,0.38,0.93,1.32,2.85
min,20.80,1505.00,19.61,65.77,3.62,33.14,0.55,0.18,2.55,1.93,5.17
25%,25.40,1864.25,38.28,81.28,7.45,53.31,1.53,0.67,4.03,3.88,9.38
50%,27.90,2231.50,44.50,85.21,9.71,61.37,1.85,0.90,4.64,4.69,10.98
75%,30.40,2675.00,54.82,88.07,13.27,70.00,2.34,1.16,5.22,5.55,12.99
max,40.80,3414.00,116.39,94.21,28.70,127.17,4.51,2.55,8.26,8.74,20.03


### 8.4 Validate Per-90 Features

Check the range, missingness and invalid values of the per-90 features before selecting metrics for the recruitment model. Any extreme rates should be examined in context before drawing conclusions.


In [ ]:
per90_columns = [f"{metric}_per90" for metric in per90_metrics]

per90_validation = pd.DataFrame({
    "min": eligible_players[per90_columns].min(),
    "median": eligible_players[per90_columns].median(),
    "max": eligible_players[per90_columns].max(),
    "missing": eligible_players[per90_columns].isna().sum(),
    "negative": (eligible_players[per90_columns] < 0).sum(),
    "infinite": np.isinf(eligible_players[per90_columns]).sum()
})

per90_validation.round(2)


### 9. Feature Selection and Correlation Analysis

Define the candidate performance features used to assess the Defensive Controller role. These combine volume, efficiency, defensive contribution and ball retention.


In [49]:
candidate_features = [
    "passes_total_per90",
    "pass_completion_pct",
    "passes_final_third_per90",
    "passes_opp_half_per90",
    "long_balls_completed_per90",
    "touches_per90",
    "possession_lost_per90",

    "tackles_per90",
    "interceptions_per90",
    "outfield_blocks_per90",
    "ball_recoveries_per90",

    "duels_won_per90",
    "duels_won_pct"
]

### 9.2 Correlation Matrix

Calculate Pearson correlations between candidate features to identify overlapping measurements. Correlation is descriptive, not causal.


In [50]:
correlation_matrix = (
    eligible_players[candidate_features]
    .corr()
    .round(2)
)

correlation_matrix

,passes_total_per90,pass_completion_pct,passes_final_third_per90,passes_opp_half_per90,long_balls_completed_per90,touches_per90,possession_lost_per90,tackles_per90,interceptions_per90,outfield_blocks_per90,ball_recoveries_per90,duels_won_per90,duels_won_pct
passes_total_per90,1.00,0.67,0.82,0.94,0.74,0.98,0.01,-0.08,0.04,-0.04,0.36,-0.20,0.25
pass_completion_pct,0.67,1.00,0.50,0.64,0.36,0.60,-0.49,-0.01,0.05,0.07,0.21,-0.26,0.26
passes_final_third_per90,0.82,0.50,1.00,0.94,0.57,0.86,0.27,-0.13,-0.23,-0.31,0.29,-0.11,0.08
passes_opp_half_per90,0.94,0.64,0.94,1.00,0.65,0.94,0.07,-0.09,-0.08,-0.17,0.32,-0.15,0.21
long_balls_completed_per90,0.74,0.36,0.57,0.65,1.00,0.73,0.17,-0.10,0.01,0.00,0.33,-0.22,0.14
touches_per90,0.98,0.60,0.86,0.94,0.73,1.00,0.17,-0.02,0.02,-0.08,0.41,-0.09,0.22
possession_lost_per90,0.01,-0.49,0.27,0.07,0.17,0.17,1.00,-0.06,-0.25,-0.36,0.20,0.19,-0.30
tackles_per90,-0.08,-0.01,-0.13,-0.09,-0.10,-0.02,-0.06,1.00,0.34,0.21,0.35,0.59,0.31
interceptions_per90,0.04,0.05,-0.23,-0.08,0.01,0.02,-0.25,0.34,1.00,0.41,0.22,0.16,0.31
outfield_blocks_per90,-0.04,0.07,-0.31,-0.17,0.00,-0.08,-0.36,0.21,0.41,1.00,-0.07,0.09,0.26


### 9.3 Flag Strong Correlations

Extract feature pairs with high absolute correlations. Review them for possible double-counting in the weighted score.


In [51]:
high_correlations = []

for i in range(len(candidate_features)):
    for j in range(i + 1, len(candidate_features)):
        feature_1 = candidate_features[i]
        feature_2 = candidate_features[j]

        correlation = correlation_matrix.loc[
            feature_1,
            feature_2
        ]

        if abs(correlation) >= 0.70:
            high_correlations.append(
                (feature_1, feature_2, correlation)
            )

high_correlations

[('passes_total_per90', 'passes_final_third_per90', np.float64(0.82)),
 ('passes_total_per90', 'passes_opp_half_per90', np.float64(0.94)),
 ('passes_total_per90', 'long_balls_completed_per90', np.float64(0.74)),
 ('passes_total_per90', 'touches_per90', np.float64(0.98)),
 ('passes_final_third_per90', 'passes_opp_half_per90', np.float64(0.94)),
 ('passes_final_third_per90', 'touches_per90', np.float64(0.86)),
 ('passes_opp_half_per90', 'touches_per90', np.float64(0.94)),
 ('long_balls_completed_per90', 'touches_per90', np.float64(0.73))]

#### Findings

The saved output flags several strong passing-volume relationships, including total passes versus opposition-half passes (**r = 0.94**) and versus final-third passes (**r = 0.82**). These suggest overlapping passing-volume signals; retaining them requires a tactical justification.


### 10. Percentile-Based Feature Scaling

Rank positive-direction metrics as percentiles within the 248-player eligible universe. This puts differently scaled football statistics onto a common 0–100 basis.


In [52]:
positive_metrics = [
    "passes_total_per90",
    "pass_completion_pct",
    "passes_final_third_per90",
    "long_balls_completed_per90",

    "tackles_per90",
    "interceptions_per90",
    "outfield_blocks_per90",
    "ball_recoveries_per90",

    "duels_won_per90",
    "duels_won_pct"
]

for metric in positive_metrics:
    eligible_players[f"{metric}_pct"] = (
        eligible_players[metric]
        .rank(pct=True)
        * 100
    )

### 10.2 Reverse Possession-Loss Direction

Reverse the possession-loss percentile so that fewer losses per 90 receive a better score. A lower event count is not always equivalent to better decision-making.


In [53]:
eligible_players["possession_lost_per90_pct"] = (
    eligible_players["possession_lost_per90"]
    .rank(
        pct=True,
        ascending=False
    )
    * 100
)

### 10.3 Inspect Percentile Transformations

Display sample players to check that the ranking direction and scale are sensible.


In [54]:
eligible_players[
    [
        "player",
        "passes_total_per90",
        "passes_total_per90_pct",
        "tackles_per90",
        "tackles_per90_pct",
        "possession_lost_per90",
        "possession_lost_per90_pct"
    ]
].sort_values(
    "passes_total_per90_pct",
    ascending=False
).head(10)

,player,passes_total_per90,passes_total_per90_pct,tackles_per90,tackles_per90_pct,possession_lost_per90,possession_lost_per90_pct
204,Vitinha,116.393211,100.000000,1.018388,4.032258,9.632249,70.161290
53,Joshua Kimmich,104.605263,99.596774,1.263158,11.693548,13.973684,18.548387
107,Aleix García,102.073991,99.193548,0.874439,1.612903,11.367713,44.758065
135,Rodri,96.245869,98.790323,2.438863,81.854839,12.075347,37.903226
287,Pedri,87.778832,98.387097,2.135738,64.919355,11.404841,43.951613
76,Manuel Locatelli,84.037949,97.983871,3.055925,93.951613,12.073901,38.306452
106,Frenkie de Jong,82.577886,97.580645,2.474038,83.467742,6.707392,96.370968
71,Maxime López,78.336735,97.177419,1.056122,4.838710,14.051020,16.532258
11,Koke,76.757603,96.774194,2.012522,61.290323,9.378354,75.000000
201,Angelo Stiller,76.320000,96.370968,1.603636,31.451613,13.876364,19.758065


### 11. Construct Tactical Dimension Scores

Combine the feature percentiles into five interpretable dimensions: defensive activity, possession control, progression, recovery and physical/duel performance.


In [55]:
eligible_players["defensive_activity_score"] = (
    eligible_players[
        [
            "tackles_per90_pct",
            "interceptions_per90_pct",
            "outfield_blocks_per90_pct"
        ]
    ].mean(axis=1)
)

eligible_players["possession_control_score"] = (
    eligible_players[
        [
            "passes_total_per90_pct",
            "pass_completion_pct_pct",
            "possession_lost_per90_pct"
        ]
    ].mean(axis=1)
)

eligible_players["progression_score"] = (
    eligible_players[
        [
            "passes_final_third_per90_pct",
            "long_balls_completed_per90_pct"
        ]
    ].mean(axis=1)
)

eligible_players["ball_recovery_score"] = (
    eligible_players["ball_recoveries_per90_pct"]
)

eligible_players["physical_duels_score"] = (
    eligible_players[
        [
            "duels_won_per90_pct",
            "duels_won_pct_pct"
        ]
    ].mean(axis=1)
)

### 11.2 Calculate Overall Controller Score

Compute the weighted Controller Score: 30% defensive activity, 25% possession control, 20% progression, 15% recovery and 10% physical/duel contribution. These weights are modelling assumptions.


In [56]:
eligible_players["controller_score"] = (
    eligible_players["defensive_activity_score"] * 0.30
    + eligible_players["possession_control_score"] * 0.25
    + eligible_players["progression_score"] * 0.20
    + eligible_players["ball_recovery_score"] * 0.15
    + eligible_players["physical_duels_score"] * 0.10
)

### 11.3 Inspect Leading Controllers

Inspect the highest-scoring players before applying recruitment constraints. This is a role-fit ranking, not a transfer recommendation.


In [57]:
top_controllers = (
    eligible_players[
        [
            "player",
            "age",
            "season_teams",
            "season_leagues",
            "minutes",
            "defensive_activity_score",
            "possession_control_score",
            "progression_score",
            "ball_recovery_score",
            "physical_duels_score",
            "controller_score"
        ]
    ]
    .sort_values(
        "controller_score",
        ascending=False
    )
    .head(20)
)

top_controllers.round(1)

,player,age,season_teams,season_leagues,minutes,defensive_activity_score,possession_control_score,progression_score,ball_recovery_score,physical_duels_score,controller_score
76,Manuel Locatelli,28.5,Juventus,Italy Serie A,3004,88.7,71.0,98.2,81.5,82.9,84.5
280,Moisés Caicedo,24.7,Chelsea,England Premier League,2802,91.5,93.1,64.9,87.1,76.4,84.4
262,Eduardo Camavinga,23.6,Real Madrid,Spain La Liga,1532,77.7,90.3,88.7,62.1,95.4,82.5
165,Aurélien Tchouaméni,26.4,Real Madrid,Spain La Liga,2622,82.9,93.7,72.4,62.9,89.7,81.2
221,Pathé Ismaël Ciss,32.3,Rayo Vallecano,Spain La Liga,2003,91.0,83.1,61.3,71.4,70.2,78.0
135,Rodri,30.0,Manchester City,England Premier League,1513,51.9,75.0,96.8,96.8,96.2,77.8
318,Mamadou Sangare,24.0,RC Lens,France Ligue 1,2327,75.3,59.1,79.8,99.6,88.7,77.1
251,Nico González,24.5,Manchester City,England Premier League,1575,76.2,84.3,72.0,70.2,79.4,76.8
290,Elliot Anderson,23.6,Nottingham Forest,England Premier League,3334,80.0,48.1,85.1,100.0,86.1,76.6
287,Pedri,23.6,FC Barcelona,Spain La Liga,2107,45.3,79.3,97.2,98.8,88.7,76.5


#### Findings

The unfiltered role-fit leaders include **Manuel Locatelli** and **Moisés Caicedo**. These rankings precede age and Premier League restrictions.


### 11.4 Rank All Eligible Players

Order the complete eligible universe by Controller Score and assign a role-fit rank.


In [58]:
role_fit_ranking = (
    eligible_players
    .sort_values(
        "controller_score",
        ascending=False
    )
    .reset_index(drop=True)
)

role_fit_ranking["role_fit_rank"] = (
    role_fit_ranking.index + 1
)

### 11.5 Review Role-Fit Ranking

Display ranked players with identifying and performance context.


In [59]:
role_fit_ranking[
    [
        "role_fit_rank",
        "player",
        "age",
        "season_teams",
        "controller_score"
    ]
].head(20)

,role_fit_rank,player,age,season_teams,controller_score
0,1,Manuel Locatelli,28.5,Juventus,84.495968
1,2,Moisés Caicedo,24.7,Chelsea,84.435484
2,3,Eduardo Camavinga,23.6,Real Madrid,82.479839
3,4,Aurélien Tchouaméni,26.4,Real Madrid,81.182796
4,5,Pathé Ismaël Ciss,32.3,Rayo Vallecano,78.044355
5,6,Rodri,30.0,Manchester City,77.802419
6,7,Mamadou Sangare,24.0,RC Lens,77.143817
7,8,Nico González,24.5,Manchester City,76.794355
8,9,Elliot Anderson,23.6,Nottingham Forest,76.646505
9,10,Pedri,23.6,FC Barcelona,76.538978


#### Findings

The saved role-fit ranking places **Manuel Locatelli first** and **Moisés Caicedo second** among the 248 eligible midfielders.


### 12. Weight Sensitivity Analysis

Define alternative weighting scenarios to test whether the ranking is dependent on one particular tactical weighting choice.


In [60]:
weight_scenarios = {
    "base": {
        "defensive_activity_score": 0.30,
        "possession_control_score": 0.25,
        "progression_score": 0.20,
        "ball_recovery_score": 0.15,
        "physical_duels_score": 0.10
    },

    "defensive_heavy": {
        "defensive_activity_score": 0.35,
        "possession_control_score": 0.20,
        "progression_score": 0.15,
        "ball_recovery_score": 0.20,
        "physical_duels_score": 0.10
    },

    "control_heavy": {
        "defensive_activity_score": 0.25,
        "possession_control_score": 0.30,
        "progression_score": 0.25,
        "ball_recovery_score": 0.10,
        "physical_duels_score": 0.10
    },

    "balanced": {
        "defensive_activity_score": 0.20,
        "possession_control_score": 0.20,
        "progression_score": 0.20,
        "ball_recovery_score": 0.20,
        "physical_duels_score": 0.20
    }
}

### 12.2 Recalculate Scenario Scores

Recalculate Controller Scores under each alternative scenario while retaining the same underlying feature and dimension definitions.


In [61]:
for scenario, weights in weight_scenarios.items():

    score = 0

    for dimension, weight in weights.items():
        score += (
            eligible_players[dimension]
            * weight
        )

    eligible_players[f"{scenario}_score"] = score

### 12.3 Calculate Scenario Ranks

Assign ranks separately for each scenario so rank movement can be compared.


In [62]:
for scenario in weight_scenarios:

    eligible_players[f"{scenario}_rank"] = (
        eligible_players[f"{scenario}_score"]
        .rank(
            ascending=False,
            method="min"
        )
    )

### 12.4 Compare Scenario Rankings

Inspect player ranks under the base, defensive-heavy, control-heavy and balanced models.


In [63]:
sensitivity_results = (
    eligible_players[
        [
            "player",
            "age",
            "season_teams",

            "base_rank",
            "defensive_heavy_rank",
            "control_heavy_rank",
            "balanced_rank",

            "base_score"
        ]
    ]
    .sort_values("base_rank")
    .head(20)
)

sensitivity_results.round(1)

,player,age,season_teams,base_rank,defensive_heavy_rank,control_heavy_rank,balanced_rank,base_score
76,Manuel Locatelli,28.5,Juventus,1.0,2.0,1.0,1.0,84.5
280,Moisés Caicedo,24.7,Chelsea,2.0,1.0,3.0,4.0,84.4
262,Eduardo Camavinga,23.6,Real Madrid,3.0,3.0,2.0,3.0,82.5
165,Aurélien Tchouaméni,26.4,Real Madrid,4.0,4.0,4.0,7.0,81.2
221,Pathé Ismaël Ciss,32.3,Rayo Vallecano,5.0,6.0,9.0,10.0,78.0
135,Rodri,30.0,Manchester City,6.0,8.0,5.0,2.0,77.8
318,Mamadou Sangare,24.0,RC Lens,7.0,7.0,11.0,6.0,77.1
251,Nico González,24.5,Manchester City,8.0,9.0,8.0,9.0,76.8
290,Elliot Anderson,23.6,Nottingham Forest,9.0,5.0,13.0,8.0,76.6
287,Pedri,23.6,FC Barcelona,10.0,11.0,6.0,5.0,76.5


#### Findings

The saved comparison shows that ranks change under alternative tactical weights. For example, Locatelli ranks first in the base scenario and second in the defensive-heavy scenario; Caicedo moves from second to first.


### 12.5 Summarise Ranking Stability

Measure how much player rankings change across scenarios; larger variation signals sensitivity to tactical assumptions.


In [64]:
rank_columns = [
    "base_rank",
    "defensive_heavy_rank",
    "control_heavy_rank",
    "balanced_rank"
]

eligible_players["mean_sensitivity_rank"] = (
    eligible_players[rank_columns]
    .mean(axis=1)
)

eligible_players["rank_range"] = (
    eligible_players[rank_columns].max(axis=1)
    - eligible_players[rank_columns].min(axis=1)
)

### 12.6 Review Rank Stability

Inspect players with the most and least stable rankings across weighting scenarios.


In [65]:
sensitivity_results = (
    eligible_players[
        [
            "player",
            "base_rank",
            "defensive_heavy_rank",
            "control_heavy_rank",
            "balanced_rank",
            "mean_sensitivity_rank",
            "rank_range"
        ]
    ]
    .sort_values("base_rank")
    .head(20)
)

sensitivity_results.round(1)

,player,base_rank,defensive_heavy_rank,control_heavy_rank,balanced_rank,mean_sensitivity_rank,rank_range
76,Manuel Locatelli,1.0,2.0,1.0,1.0,1.2,1.0
280,Moisés Caicedo,2.0,1.0,3.0,4.0,2.5,3.0
262,Eduardo Camavinga,3.0,3.0,2.0,3.0,2.8,1.0
165,Aurélien Tchouaméni,4.0,4.0,4.0,7.0,4.8,3.0
221,Pathé Ismaël Ciss,5.0,6.0,9.0,10.0,7.5,5.0
135,Rodri,6.0,8.0,5.0,2.0,5.2,6.0
318,Mamadou Sangare,7.0,7.0,11.0,6.0,7.8,5.0
251,Nico González,8.0,9.0,8.0,9.0,8.5,1.0
290,Elliot Anderson,9.0,5.0,13.0,8.0,8.8,8.0
287,Pedri,10.0,11.0,6.0,5.0,8.0,6.0


### 13. Apply Recruitment Feasibility Filters

Filter the performance-ranked universe to players aged 28 or younger and exclude players with Premier League involvement during the season. These are case-study recruitment assumptions, not footballing performance measures.


In [70]:
recruitment_pool = role_fit_ranking[
    (role_fit_ranking["age"] <= 28)
    &
    (
        ~role_fit_ranking["season_leagues"]
        .str.contains(
            "England Premier League",
            na=False
        )
    )
].copy()

### 13.2 Check Eligible Pool Size

Compare the full role-fit universe against the pool remaining after recruitment constraints.


In [71]:
print(
    "Role-fit universe:",
    len(role_fit_ranking)
)

print(
    "Recruitment-eligible:",
    len(recruitment_pool)
)

Role-fit universe: 248
Recruitment-eligible: 95


#### Findings

The recruitment filters reduce the player universe from **248** to **95** candidates. This is a feasibility-filtered subset, not a separate performance model.


### 13.3 Re-rank Recruitment Candidates

Assign recruitment-specific ranks after filtering; these differ from ranks within the original 248-player performance universe.


In [72]:
recruitment_pool = (
    recruitment_pool
    .sort_values(
        "controller_score",
        ascending=False
    )
    .reset_index(drop=True)
)

recruitment_pool["recruitment_rank"] = (
    recruitment_pool.index + 1
)

### 13.4 Inspect Recruitment Ranking

Review the best eligible recruitment candidates and their role-fit ranks.


In [73]:
recruitment_pool[
    [
        "recruitment_rank",
        "role_fit_rank",
        "player",
        "age",
        "season_teams",
        "season_leagues",
        "minutes",
        "market_value",
        "market_value_currency",
        "controller_score"
    ]
].head(20).round(1)

,recruitment_rank,role_fit_rank,player,age,season_teams,season_leagues,minutes,market_value,market_value_currency,controller_score
0,1,3,Eduardo Camavinga,23.6,Real Madrid,Spain La Liga,1532,48000000.0,EUR,82.5
1,2,4,Aurélien Tchouaméni,26.4,Real Madrid,Spain La Liga,2622,73000000.0,EUR,81.2
2,3,7,Mamadou Sangare,24.0,RC Lens,France Ligue 1,2327,39000000.0,EUR,77.1
3,4,10,Pedri,23.6,FC Barcelona,Spain La Liga,2107,164000000.0,EUR,76.5
4,5,13,Éderson,27.0,Atalanta,Italy Serie A,2235,43000000.0,EUR,73.6
5,6,15,Jobe Bellingham,20.8,Borussia Dortmund,Germany Bundesliga,1800,38000000.0,EUR,72.6
6,7,16,Vitinha,26.4,Paris Saint-Germain,France Ligue 1,2121,130000000.0,EUR,72.1
7,8,18,Tom Krauß,25.0,1. FC Köln,Germany Bundesliga,1801,4700000.0,EUR,70.9
8,9,19,Pepelu,27.9,Valencia,Spain La Liga,2232,7300000.0,EUR,70.6
9,10,20,Cristian Cásseres Jr.,26.4,Toulouse,France Ligue 1,2499,8199999.0,EUR,70.3


#### Findings

The saved recruitment ranking begins with **Eduardo Camavinga** (overall role-fit rank 3) and **Aurélien Tchouaméni** (overall role-fit rank 4).


### 14. Define a Performance Shortlist

Summarise the Controller Score distribution among recruitment-eligible players before choosing a shortlist cutoff.


In [74]:
recruitment_pool["controller_score"].describe().round(2)

count    95.00
mean     48.86
std      14.60
min      16.24
25%      37.99
50%      50.55
75%      58.52
max      82.48
Name: controller_score, dtype: float64

#### Findings

Among the **95** recruitment-eligible players, Controller Score has mean **48.86**, median **50.55**, minimum **16.24** and maximum **82.48**.


### 14.2 Examine Score Percentiles

Compare candidate-score quantiles to support a transparent shortlist threshold.


In [75]:
recruitment_pool[
    "controller_score"
].quantile(
    [0.50, 0.75, 0.80, 0.90, 0.95]
).round(2)

0.50    50.55
0.75    58.52
0.80    59.88
0.90    70.05
0.95    72.91
Name: controller_score, dtype: float64

#### Findings

The recruitment-pool score percentiles are **50th: 50.55**, **75th: 58.52**, **80th: 59.88**, **90th: 70.05**, and **95th: 72.91**.


### 14.3 Set the Shortlist Threshold

Use the 90th percentile of the recruitment pool’s Controller Score distribution as the performance shortlist cutoff.


In [76]:
shortlist_threshold = (
    recruitment_pool["controller_score"]
    .quantile(0.90)
)

print(
    "90th percentile threshold:",
    round(shortlist_threshold, 2)
)

90th percentile threshold: 70.05


#### Findings

The saved **90th-percentile cutoff is 70.05 Controller Score points**.


### 14.4 Create Performance Shortlist

Retain candidates meeting or exceeding the threshold and rank them by Controller Score.


In [77]:
performance_shortlist = recruitment_pool[
    recruitment_pool["controller_score"]
    >= shortlist_threshold
].copy()

performance_shortlist = (
    performance_shortlist
    .sort_values(
        "controller_score",
        ascending=False
    )
    .reset_index(drop=True)
)

performance_shortlist["shortlist_rank"] = (
    performance_shortlist.index + 1
)

### 14.5 Inspect Performance Shortlist

Review the shortlisted candidates with their role-fit and recruitment context.


In [78]:
performance_shortlist[
    [
        "shortlist_rank",
        "player",
        "age",
        "season_teams",
        "season_leagues",
        "minutes",
        "controller_score",
        "market_value",
        "market_value_currency"
    ]
].round(1)

,shortlist_rank,player,age,season_teams,season_leagues,minutes,controller_score,market_value,market_value_currency
0,1,Eduardo Camavinga,23.6,Real Madrid,Spain La Liga,1532,82.5,48000000.0,EUR
1,2,Aurélien Tchouaméni,26.4,Real Madrid,Spain La Liga,2622,81.2,73000000.0,EUR
2,3,Mamadou Sangare,24.0,RC Lens,France Ligue 1,2327,77.1,39000000.0,EUR
3,4,Pedri,23.6,FC Barcelona,Spain La Liga,2107,76.5,164000000.0,EUR
4,5,Éderson,27.0,Atalanta,Italy Serie A,2235,73.6,43000000.0,EUR
5,6,Jobe Bellingham,20.8,Borussia Dortmund,Germany Bundesliga,1800,72.6,38000000.0,EUR
6,7,Vitinha,26.4,Paris Saint-Germain,France Ligue 1,2121,72.1,130000000.0,EUR
7,8,Tom Krauß,25.0,1. FC Köln,Germany Bundesliga,1801,70.9,4700000.0,EUR
8,9,Pepelu,27.9,Valencia,Spain La Liga,2232,70.6,7300000.0,EUR
9,10,Cristian Cásseres Jr.,26.4,Toulouse,France Ligue 1,2499,70.3,8199999.0,EUR


#### Findings

The saved shortlist ranks **Camavinga** first and **Tchouaméni** second. The shortlist is selected from the 95-player recruitment pool, not directly from all 248 midfielders.


### 15. Compare Shortlisted Player Profiles

Collect shortlist-level tactical dimension scores for side-by-side evaluation.


In [79]:
profile_columns = [
    "shortlist_rank",
    "player",
    "age",
    "season_teams",
    "controller_score",
    "defensive_activity_score",
    "possession_control_score",
    "progression_score",
    "ball_recovery_score",
    "physical_duels_score",
    "market_value"
]

shortlist_profiles = (
    performance_shortlist[profile_columns]
    .copy()
)

shortlist_profiles.round(1)

,shortlist_rank,player,age,season_teams,controller_score,defensive_activity_score,possession_control_score,progression_score,ball_recovery_score,physical_duels_score,market_value
0,1,Eduardo Camavinga,23.6,Real Madrid,82.5,77.7,90.3,88.7,62.1,95.4,48000000.0
1,2,Aurélien Tchouaméni,26.4,Real Madrid,81.2,82.9,93.7,72.4,62.9,89.7,73000000.0
2,3,Mamadou Sangare,24.0,RC Lens,77.1,75.3,59.1,79.8,99.6,88.7,39000000.0
3,4,Pedri,23.6,FC Barcelona,76.5,45.3,79.3,97.2,98.8,88.7,164000000.0
4,5,Éderson,27.0,Atalanta,73.6,66.8,77.2,77.4,80.6,66.9,43000000.0
5,6,Jobe Bellingham,20.8,Borussia Dortmund,72.6,80.0,67.1,59.1,86.3,70.8,38000000.0
6,7,Vitinha,26.4,Paris Saint-Germain,72.1,39.1,89.9,99.4,89.9,45.4,130000000.0
7,8,Tom Krauß,25.0,1. FC Köln,70.9,89.2,56.9,34.9,94.8,86.9,4700000.0
8,9,Pepelu,27.9,Valencia,70.6,65.6,65.7,74.2,87.5,64.9,7300000.0
9,10,Cristian Cásseres Jr.,26.4,Toulouse,70.3,76.5,46.8,68.1,97.6,74.2,8199999.0


### 15.2 Identify Dimension Strengths

Identify each shortlisted player’s strongest tactical dimension. This is relative to the model’s dimensions, not an independent scouting assessment.


In [80]:
dimension_columns = {
    "Defensive Activity": "defensive_activity_score",
    "Possession Control": "possession_control_score",
    "Progression": "progression_score",
    "Ball Recovery": "ball_recovery_score",
    "Physical / Duels": "physical_duels_score"
}

def strongest_dimension(row):
    return max(
        dimension_columns,
        key=lambda name: row[dimension_columns[name]]
    )

def weakest_dimension(row):
    return min(
        dimension_columns,
        key=lambda name: row[dimension_columns[name]]
    )

shortlist_profiles["strongest_dimension"] = (
    shortlist_profiles.apply(
        strongest_dimension,
        axis=1
    )
)

shortlist_profiles["weakest_dimension"] = (
    shortlist_profiles.apply(
        weakest_dimension,
        axis=1
    )
)

### 15.3 Review Tactical Profiles

Inspect the balance of defensive, control, progression, recovery and physical contributions across shortlisted players.


In [81]:
shortlist_profiles[
    [
        "player",
        "controller_score",
        "defensive_activity_score",
        "possession_control_score",
        "progression_score",
        "ball_recovery_score",
        "physical_duels_score",
        "strongest_dimension",
        "weakest_dimension"
    ]
].round(1)

,player,controller_score,defensive_activity_score,possession_control_score,progression_score,ball_recovery_score,physical_duels_score,strongest_dimension,weakest_dimension
0,Eduardo Camavinga,82.5,77.7,90.3,88.7,62.1,95.4,Physical / Duels,Ball Recovery
1,Aurélien Tchouaméni,81.2,82.9,93.7,72.4,62.9,89.7,Possession Control,Ball Recovery
2,Mamadou Sangare,77.1,75.3,59.1,79.8,99.6,88.7,Ball Recovery,Possession Control
3,Pedri,76.5,45.3,79.3,97.2,98.8,88.7,Ball Recovery,Defensive Activity
4,Éderson,73.6,66.8,77.2,77.4,80.6,66.9,Ball Recovery,Defensive Activity
5,Jobe Bellingham,72.6,80.0,67.1,59.1,86.3,70.8,Ball Recovery,Progression
6,Vitinha,72.1,39.1,89.9,99.4,89.9,45.4,Progression,Defensive Activity
7,Tom Krauß,70.9,89.2,56.9,34.9,94.8,86.9,Ball Recovery,Progression
8,Pepelu,70.6,65.6,65.7,74.2,87.5,64.9,Ball Recovery,Physical / Duels
9,Cristian Cásseres Jr.,70.3,76.5,46.8,68.1,97.6,74.2,Ball Recovery,Possession Control


### 16. Market-Value Context

Express estimated market values in millions of euros for more readable comparisons. Estimated values are not verified transfer fees.


In [82]:
performance_shortlist["market_value_m"] = (
    performance_shortlist["market_value"] / 1_000_000
)

### 16.2 Compare Shortlist Valuations

Sort shortlisted players by estimated market value to show the range of valuations within the performance shortlist.


In [83]:
performance_shortlist[
    [
        "player",
        "controller_score",
        "market_value_m"
    ]
].sort_values("market_value_m")

,player,controller_score,market_value_m
7,Tom Krauß,70.866935,4.700000
8,Pepelu,70.564516,7.300000
9,Cristian Cásseres Jr.,70.322581,8.199999
5,Jobe Bellingham,72.614247,38.000000
2,Mamadou Sangare,77.143817,39.000000
4,Éderson,73.602151,43.000000
0,Eduardo Camavinga,82.479839,48.000000
1,Aurélien Tchouaméni,81.182796,73.000000
6,Vitinha,72.116935,130.000000
3,Pedri,76.538978,164.000000


#### Findings

The saved valuation table identifies **Tom Krauß (€4.7m)** and **Pepelu (€7.3m)** among the lowest estimated market values on the performance shortlist.


### 16.3 Prepare Recruitment-Pool Values

Convert estimated market values to millions of euros for the complete recruitment pool.


In [84]:
recruitment_pool["market_value_m"] = (
    recruitment_pool["market_value"] / 1_000_000
)

### 16.4 Validate Valuation Coverage

Check the availability, currency and distribution of market-value estimates before using them in analysis.


In [85]:
print(
    "Players:",
    len(recruitment_pool)
)

print(
    "Missing market values:",
    recruitment_pool["market_value_m"].isna().sum()
)

print(
    recruitment_pool["market_value_currency"]
    .value_counts(dropna=False)
)

recruitment_pool["market_value_m"].describe().round(2)

Players: 95
Missing market values: 0
market_value_currency
EUR    95
Name: count, dtype: int64


count     95.00
mean      22.43
std       29.08
min        1.50
25%        6.65
50%       12.40
75%       26.50
max      164.00
Name: market_value_m, dtype: float64

#### Findings

All **95** recruitment candidates have a non-missing market-value estimate, recorded in **EUR**. Mean value is **€22.43m** and median **€12.40m**, indicating a right-skewed distribution.


### 16.5 Examine Score–Value Correlation

Calculate the association between performance-based Controller Score and estimated market value. Correlation alone cannot identify underpriced players.


In [86]:
recruitment_pool[
    [
        "controller_score",
        "market_value_m"
    ]
].corr().round(2)

,controller_score,market_value_m
controller_score,1.00,0.42
market_value_m,0.42,1.00


#### Findings

The saved Pearson correlation between Controller Score and estimated market value is **0.42**, a moderate positive sample association.


### 17. Value-Adjusted Regression Analysis

Prepare a regression dataset and log-transform market value to reduce skew. The model estimates the average relationship between valuation and Controller Score in this sample.


In [88]:
import numpy as np
from sklearn.linear_model import LinearRegression

value_df = recruitment_pool[
    [
        "player",
        "market_value_m",
        "controller_score"
    ]
].copy()

value_df["log_market_value"] = np.log1p(
    value_df["market_value_m"]
)

### 17.2 Fit Linear Regression

Fit a simple linear regression predicting Controller Score from log-transformed market value. This is a descriptive benchmark, not a transfer-price prediction model.


In [89]:
X = value_df[["log_market_value"]]
y = value_df["controller_score"]

value_model = LinearRegression()

value_model.fit(X, y)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](1,)",[5.83]
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](1,)",['log_market_value']
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,33.23
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,1
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(1)


### 17.3 Estimate Expected Controller Score

Generate each player’s expected Controller Score from the fitted market-value relationship.


In [90]:
value_df["expected_controller_score"] = (
    value_model.predict(X)
)

### 17.4 Calculate Value Residuals

Subtract expected score from actual score. Positive residuals indicate better performance-model scores than the valuation relationship predicts, not proven bargains.


In [91]:
value_df["value_residual"] = (
    value_df["controller_score"]
    - value_df["expected_controller_score"]
)

### 17.5 Attach Value Indicators

Merge expected scores and residuals into the recruitment pool for comparison.


In [92]:
recruitment_pool = recruitment_pool.merge(
    value_df[
        [
            "player",
            "expected_controller_score",
            "value_residual"
        ]
    ],
    on="player",
    how="left"
)

### 17.6 Review Positive Residuals

Rank players by residual to identify candidates who stand out relative to this simple valuation benchmark.


In [93]:
recruitment_pool[
    [
        "player",
        "age",
        "season_teams",
        "controller_score",
        "market_value_m",
        "expected_controller_score",
        "value_residual"
    ]
].sort_values(
    "value_residual",
    ascending=False
).head(15).round(1)

,player,age,season_teams,controller_score,market_value_m,expected_controller_score,value_residual
7,Tom Krauß,25.0,1. FC Köln,70.9,4.7,43.4,27.5
0,Eduardo Camavinga,23.6,Real Madrid,82.5,48.0,55.9,26.6
8,Pepelu,27.9,Valencia,70.6,7.3,45.6,25.0
9,Cristian Cásseres Jr.,26.4,Toulouse,70.3,8.2,46.2,24.1
1,Aurélien Tchouaméni,26.4,Real Madrid,81.2,73.0,58.3,22.9
10,Antonio Blanco,25.9,Deportivo Alavés,69.6,9.7,47.1,22.6
2,Mamadou Sangare,24.0,RC Lens,77.1,39.0,54.7,22.4
15,Manu Morlanes,27.5,Mallorca,61.4,2.2,40.0,21.3
14,Noah Cadiou,27.7,Lorient,61.9,3.1,41.5,20.4
11,Charles Vanhoutte,27.8,Nice,64.3,6.8,45.2,19.1


#### Findings

The saved residual ranking places **Tom Krauß** and **Eduardo Camavinga** among the leading positive residuals. Residuals measure relative model fit, not actual transfer affordability.


### 17.7 Evaluate Regression Fit

Report in-sample R² to show how much variation in Controller Score is explained by the log-market-value model.


In [94]:
print(
    "R²:",
    round(
        value_model.score(X, y),
        3
    )
)

R²: 0.137


#### Findings

The fitted model has an in-sample **R² of 0.137**. Log market value explains only **13.7%** of Controller Score variation in this recruitment pool; the residual analysis is therefore exploratory.


### 18. Combine Performance and Value Evidence

Attach the value residuals to the performance shortlist to compare footballing suitability and estimated valuation.


In [95]:
shortlist_final = performance_shortlist.merge(
    value_df[
        [
            "player",
            "expected_controller_score",
            "value_residual"
        ]
    ],
    on="player",
    how="left"
)

### 18.2 Inspect Shortlist Value Comparison

Review Controller Score, market value and residual together for each shortlisted candidate.


In [96]:
shortlist_final[
    [
        "player",
        "age",
        "controller_score",
        "market_value_m",
        "value_residual",
        "defensive_activity_score",
        "possession_control_score",
        "progression_score",
        "ball_recovery_score",
        "physical_duels_score"
    ]
].sort_values(
    "controller_score",
    ascending=False
).round(1)

,player,age,controller_score,market_value_m,value_residual,defensive_activity_score,possession_control_score,progression_score,ball_recovery_score,physical_duels_score
0,Eduardo Camavinga,23.6,82.5,48.0,26.6,77.7,90.3,88.7,62.1,95.4
1,Aurélien Tchouaméni,26.4,81.2,73.0,22.9,82.9,93.7,72.4,62.9,89.7
2,Mamadou Sangare,24.0,77.1,39.0,22.4,75.3,59.1,79.8,99.6,88.7
3,Pedri,23.6,76.5,164.0,13.5,45.3,79.3,97.2,98.8,88.7
4,Éderson,27.0,73.6,43.0,18.3,66.8,77.2,77.4,80.6,66.9
5,Jobe Bellingham,20.8,72.6,38.0,18.0,80.0,67.1,59.1,86.3,70.8
6,Vitinha,26.4,72.1,130.0,10.5,39.1,89.9,99.4,89.9,45.4
7,Tom Krauß,25.0,70.9,4.7,27.5,89.2,56.9,34.9,94.8,86.9
8,Pepelu,27.9,70.6,7.3,25.0,65.6,65.7,74.2,87.5,64.9
9,Cristian Cásseres Jr.,26.4,70.3,8.2,24.1,76.5,46.8,68.1,97.6,74.2


#### Findings

The saved table compares the shortlist’s Controller Scores, estimated market values and residuals. The value estimates should not be treated as club asking prices.


### 19. Preliminary Transfer Feasibility Assessment

Assign qualitative feasibility tiers to shortlisted players. These tiers are provisional judgements and should not be presented as verified club availability or transfer prices.


In [97]:
feasibility_data = {
    "player": [
        "Eduardo Camavinga",
        "Aurélien Tchouaméni",
        "Mamadou Sangare",
        "Pedri",
        "Éderson",
        "Jobe Bellingham",
        "Vitinha",
        "Tom Krauß",
        "Pepelu",
        "Cristian Cásseres Jr."
    ],
    "feasibility_tier": [
        "Very Difficult",
        "Very Difficult",
        "Very Difficult",
        "Very Difficult",
        "Difficult",
        "Difficult",
        "Very Difficult",
        "Potential Opportunity",
        "Potential Opportunity",
        "Potential Opportunity"
    ]
}

feasibility_df = pd.DataFrame(feasibility_data)

### 19.2 Merge Feasibility Assessment

Join the qualitative feasibility labels to the performance and value shortlist.


In [98]:
final_shortlist = shortlist_final.merge(
    feasibility_df,
    on="player",
    how="left"
)

### 19.3 Final Recruitment Decision Table

Present a consolidated comparison of performance fit, age, team, valuation and provisional feasibility. This is a decision-support table, not a final scouting verdict.


In [99]:
final_decision_table = final_shortlist[
    [
        "player",
        "age",
        "season_teams",
        "controller_score",
        "market_value_m",
        "value_residual",
        "defensive_activity_score",
        "possession_control_score",
        "progression_score",
        "ball_recovery_score",
        "physical_duels_score",
        "feasibility_tier"
    ]
].sort_values(
    "controller_score",
    ascending=False
)

final_decision_table.round(1)

,player,age,season_teams,controller_score,market_value_m,value_residual,defensive_activity_score,possession_control_score,progression_score,ball_recovery_score,physical_duels_score,feasibility_tier
0,Eduardo Camavinga,23.6,Real Madrid,82.5,48.0,26.6,77.7,90.3,88.7,62.1,95.4,Very Difficult
1,Aurélien Tchouaméni,26.4,Real Madrid,81.2,73.0,22.9,82.9,93.7,72.4,62.9,89.7,Very Difficult
2,Mamadou Sangare,24.0,RC Lens,77.1,39.0,22.4,75.3,59.1,79.8,99.6,88.7,Very Difficult
3,Pedri,23.6,FC Barcelona,76.5,164.0,13.5,45.3,79.3,97.2,98.8,88.7,Very Difficult
4,Éderson,27.0,Atalanta,73.6,43.0,18.3,66.8,77.2,77.4,80.6,66.9,Difficult
5,Jobe Bellingham,20.8,Borussia Dortmund,72.6,38.0,18.0,80.0,67.1,59.1,86.3,70.8,Difficult
6,Vitinha,26.4,Paris Saint-Germain,72.1,130.0,10.5,39.1,89.9,99.4,89.9,45.4,Very Difficult
7,Tom Krauß,25.0,1. FC Köln,70.9,4.7,27.5,89.2,56.9,34.9,94.8,86.9,Potential Opportunity
8,Pepelu,27.9,Valencia,70.6,7.3,25.0,65.6,65.7,74.2,87.5,64.9,Potential Opportunity
9,Cristian Cásseres Jr.,26.4,Toulouse,70.3,8.2,24.1,76.5,46.8,68.1,97.6,74.2,Potential Opportunity


#### Findings

The final decision table combines the quantitative shortlist with qualitative feasibility tiers. These labels require external scouting and transfer-market validation before being treated as actionable recommendations.


### 20. Export Processed Datasets

Create the processed-data directory for reproducible outputs.


In [100]:
PROCESSED_DATA_DIR = (
    PROJECT_ROOT / "data" / "processed"
)

PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

### 20.2 Export Full Eligible Dataset

Save all 248 playing-time-eligible midfielders with engineered performance features.


In [101]:
eligible_players.to_csv(
    PROCESSED_DATA_DIR
    / "defensive_controller_candidates_2025_26.csv",
    index=False
)

### 20.3 Export Recruitment Pool

Save the recruitment-filtered player pool and its scoring and value-analysis fields.


In [102]:
recruitment_pool.to_csv(
    PROCESSED_DATA_DIR
    / "recruitment_pool_2025_26.csv",
    index=False
)

### 20.4 Export Final Shortlist

Save the consolidated shortlist decision table.


In [103]:
final_decision_table.to_csv(
    PROCESSED_DATA_DIR
    / "defensive_controller_shortlist_2025_26.csv",
    index=False
)

### 20.5 Verify Exported Files

List the output files to confirm that the intended CSV deliverables were written.


In [104]:
for file in PROCESSED_DATA_DIR.glob("*.csv"):
    print(file.name)

defensive_controller_candidates_2025_26.csv
defensive_controller_shortlist_2025_26.csv
recruitment_pool_2025_26.csv


#### Findings

The saved output lists three CSVs: `defensive_controller_candidates_2025_26.csv`, `recruitment_pool_2025_26.csv`, and `defensive_controller_shortlist_2025_26.csv`.
